In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv('data/spam.csv',encoding='latin-1')

In [3]:
df.shape

(5572, 5)

In [4]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [6]:
df=df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1) 

In [7]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
df['v1'].value_counts()

v1
ham     4825
spam     747
Name: count, dtype: int64

In [9]:
df.rename(columns={'v1': 'label','v2': 'msg'}, inplace=True)

C:\Users\Haidi\AppData\Local\Temp\ipykernel_4684\1592899935.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'v1': 'label','v2': 'msg'}, inplace=True)


In [10]:
df.head()

,label,msg
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
df.duplicated().sum()

np.int64(403)

In [12]:
df.drop_duplicates(inplace=True)

C:\Users\Haidi\AppData\Local\Temp\ipykernel_4684\3006716147.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(inplace=True)


In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
from text_utils import clean_text

raw_sample = df['msg'].head(5).copy()
df['msg'] = df['msg'].apply(clean_text)

for before, after in zip(raw_sample, df['msg'].head(5)):
    print('BEFORE:', before)
    print('AFTER :', after, '\n')

BEFORE: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
AFTER : go jurong point crazy available bugis n great world la e buffet cine got amore wat 

BEFORE: Ok lar... Joking wif u oni...
AFTER : ok lar joking wif u oni 

BEFORE: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
AFTER : free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ratetcs apply 

BEFORE: U dun say so early hor... U c already then say...
AFTER : u dun say early hor u c already say 

BEFORE: Nah I don't think he goes to usf, he lives around here though
AFTER : nah dont think goes usf lives around though 



In [15]:
n_empty = (df['msg'] == '').sum()
print('Empty messages after cleaning:', n_empty)
df = df[df['msg'] != '']

Empty messages after cleaning: 6


In [16]:
print('Duplicate rows after cleaning:', df.duplicated().sum())
df = df.drop_duplicates()

print('Texts with conflicting labels:', (df.groupby('msg')['label'].nunique() > 1).sum())

df = df.reset_index(drop=True)
print('Final shape:', df.shape)
print(df['label'].value_counts())
print(df.isnull().sum())

Duplicate rows after cleaning: 100
Texts with conflicting labels: 0
Final shape: (5063, 2)
label
ham     4481
spam     582
Name: count, dtype: int64
label    0
msg      0
dtype: int64


In [17]:
df.to_csv('data/cleaned_spam.csv', index=False)

In [18]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])

train_df.to_csv('data/train.csv', index=False)
test_df.to_csv('data/test.csv', index=False)